In [67]:
import pandas as pd
import numpy as np
import os

In [68]:
df = pd.read_csv('../data/processed/car_clean.csv')

print(f'Data shape: {df.shape}')
print('\n............................. Data columns ..........................')
print(df.columns.tolist())

Data shape: (298, 9)

............................. Data columns ..........................
['Car_Name', 'Year', 'Selling_Price', 'Present_Price', 'Driven_kms', 'Fuel_Type', 'Selling_type', 'Transmission', 'Owner']


In [69]:
# Create Car_Age and Resolve Collinearity

# Create Car_Age
df['Car_Age'] = df['Year'].max() - df['Year']
print(df[['Year', 'Car_Age']].head())

# Resolove collinearity
df.drop(columns='Year', inplace=True)
print(df.columns)



   Year  Car_Age
0  2014        4
1  2013        5
2  2017        1
3  2011        7
4  2014        4
Index(['Car_Name', 'Selling_Price', 'Present_Price', 'Driven_kms', 'Fuel_Type',
       'Selling_type', 'Transmission', 'Owner', 'Car_Age'],
      dtype='str')


#### Create Car_Age and Resolve Collinearity

A new feature, Car_Age, was created by subtracting the manufacturing year from the dataset's reference year (2018). This feature is more meaningful for predicting vehicle prices because depreciation depends on how old a car is rather than the calendar year it was manufactured.

Since Year and Car_Age are perfectly inversely correlated, retaining both would introduce redundant information and perfect multicollinearity. Therefore, the Year column was removed, and Car_Age was retained as the primary feature representing vehicle age.

In [70]:
# Encode Selling_type and Transmission

df['Selling_type'] = np.where(df['Selling_type'] == 'Dealer', 0,1)
df['Transmission'] = np.where(df['Transmission'] == 'Manual', 0,1)

df[['Selling_type', 'Transmission']].head()

,Selling_type,Transmission
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


In [71]:
df = pd.get_dummies(df, columns=['Fuel_Type'], dtype=int)

print(df.head())

  Car_Name  Selling_Price  Present_Price  Driven_kms  Selling_type  \
0     ritz           3.35           5.59       27000             0   
1      sx4           4.75           9.54       43000             0   
2     ciaz           7.25           9.85        6900             0   
3  wagon r           2.85           4.15        5200             0   
4    swift           4.60           6.87       42450             0   

   Transmission  Owner  Car_Age  Fuel_Type_CNG  Fuel_Type_Diesel  \
0             0      0        4              0                 0   
1             0      0        5              0                 1   
2             0      0        1              0                 0   
3             0      0        7              0                 0   
4             0      0        4              0                 1   

   Fuel_Type_Petrol  
0                 1  
1                 0  
2                 1  
3                 1  
4                 0  


In [72]:
df['Brand'] = df['Car_Name'].apply(lambda x: x.split()[0])

# Frequency encode Brand
brand_freq = df['Brand'].value_counts()
df['Brand_Freq'] = df['Brand'].map(brand_freq)

print(df[['Brand', 'Brand_Freq']].head(10))

# Check whether Brand_Freq actually correlates with Selling_Price
corr = df['Brand_Freq'].corr(df['Selling_Price'])
print(f'\nCorrelation between Brand_Freq and Selling_Price: {corr}')

# Groupby view - average selling price by brand, sorted by frequency
brand_summary = df.groupby('Brand').agg(
    freq=('Brand', 'count'),
    avg_price=('Selling_Price', 'mean')
).sort_values('freq', ascending=False)

print(brand_summary)



    Brand  Brand_Freq
0    ritz           4
1     sx4           6
2    ciaz           9
3   wagon           4
4   swift           5
5  vitara           1
6    ciaz           9
7       s           1
8    ciaz           9
9    ciaz           9



Correlation between Brand_Freq and Selling_Price: -0.11580615768823799
          freq  avg_price
Brand                    
city        26   7.419231
Bajaj       25   0.528000
Royal       17   1.144706
Honda       17   0.438235
corolla     17   6.848824
Hero        15   0.362000
verna       14   6.107857
etios       11   4.204545
brio        10   4.745000
fortuner    10  18.254000
ciaz         9   7.472222
i20          9   4.766667
innova       9  12.777778
grand        8   4.943750
Yamaha       8   0.582500
TVS          8   0.462500
amaze        7   4.221429
jazz         7   5.828571
sx4          6   3.158333
alto         6   2.616667
eon          6   2.900000
i10          5   3.060000
swift        5   4.540000
ertiga       5   6.580000
dzire        4   4.475000
ritz         4   2.862500
KTM          4   1.337500
wagon        4   2.512500
xcent        3   4.966667
creta        3  11.800000
elantra      2  11.600000
Activa       2   0.425000
800          1   0.350000
Suzuki       1   

#### Drop Car_Name, Brand, Brand_Freq

Car_Name has 98 unique values across 298 rows, far too sparse to one-hot encode
without severely overfitting. Brand was extracted by taking the first word of
Car_Name to reduce cardinality to 44 categories, then frequency-encoded.

Both were dropped after inspection showed the feature wasn't doing what it was
intended to do. Splitting on the first word doesn't reliably extract manufacturer —
for single-word models (ritz, sx4, ciaz) it captures the model name itself, and the
resulting groups mix two entirely different vehicle types: cars (city, corolla,
verna, fortuner — avg_price 5-18 Lakhs) and two-wheelers (Bajaj, Royal Enfield,
Hero, Honda, TVS, Yamaha — avg_price 0.4-1.1 Lakhs). This lines up with the Activa
3G finding from data cleaning — the dataset mixes cars and motorcycles/scooters,
which "Brand" as constructed doesn't cleanly separate.

Brand_Freq's correlation with Selling_Price was also weak (-0.1158), confirming it
carries little signal beyond what Present_Price already captures — Present_Price
is continuous and directly reflects vehicle type and market segment, making it a
stronger and more precise substitute. All three columns were dropped.

In [73]:
df = df.drop(columns=['Car_Name', 'Brand', 'Brand_Freq'])
print("Columns remaining:", df.columns.tolist())
print("Shape:", df.shape)

Columns remaining: ['Selling_Price', 'Present_Price', 'Driven_kms', 'Selling_type', 'Transmission', 'Owner', 'Car_Age', 'Fuel_Type_CNG', 'Fuel_Type_Diesel', 'Fuel_Type_Petrol']
Shape: (298, 10)


In [74]:
print("----- Final feature check ----")
print(f"Shape: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nAll clean: {df.isnull().sum().sum() == 0}")
print(f"\nDtypes:\n{df.dtypes}")

----- Final feature check ----
Shape: (298, 10)
Missing values:
Series([], dtype: int64)

All clean: True

Dtypes:
Selling_Price       float64
Present_Price       float64
Driven_kms            int64
Selling_type          int64
Transmission          int64
Owner                 int64
Car_Age               int64
Fuel_Type_CNG         int64
Fuel_Type_Diesel      int64
Fuel_Type_Petrol      int64
dtype: object


In [75]:
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/car_features.csv', index=False)

print('Saved to ../data/processed/car_features.csv')
print(f"File size: {os.path.getsize('../data/processed/car_features.csv')}")

Saved to ../data/processed/car_features.csv
File size: 9211


## Feature Engineering Summary

### Car_Age (1 new column)
- Created as 2018 - Year (dataset's reference year)
- Year dropped afterward — perfectly collinear with Car_Age, keeping both would
  introduce redundant information

### Binary encoding (2 columns)
- Selling_type: Dealer = 0, Individual = 1
- Transmission: Manual = 0, Automatic = 1

### Fuel_Type (one-hot, 3 columns)
- Fuel_Type_CNG, Fuel_Type_Diesel, Fuel_Type_Petrol

### Car_Name / Brand dropped
- Car_Name (98 unique values) too sparse to one-hot encode on 298 rows
- Brand (44 categories, first word of Car_Name) and Brand_Freq (frequency encoding)
  were tested but dropped — the grouping conflated cars and two-wheelers rather
  than capturing true manufacturer identity, and Brand_Freq's correlation with
  Selling_Price was weak (-0.1158). Present_Price already captures vehicle
  type/segment more precisely.

### Owner
- Left unchanged — already numeric and ordinal (0/1/3 = first/second/fourth owner)

### Final dataset
- 298 rows x 10 columns, 0 missing values, all numeric
- Saved to `data/processed/car_features.csv`

### Next step
Modelling — baseline model, then tuning with a gradient-boosted model (LightGBM).